# 2 · Evaluating a strategy

Is my strategy any good, and how would I know?

The short answer is that a single run cannot tell you. This notebook builds
up to the comparison that can.

In [1]:
import tradefloor as tf

universe = tf.Universe.random(40, seed=111)
print("universe:", len(universe), "instruments")

universe: 40 instruments


## Writing a strategy down

`StrategySpec` records a strategy as a declarative, versioned, hashable
document rather than a Python callable. The reason is citation: a reader can
re-run your seed and get your market, but there is no way to hand them a
callable. A spec they can.

The grammar is small on purpose. Five behavioural parts: a signal, a
concentration (`top_k`), an exposure (`gross`), a participation cap, and
`execution.cadence` -- how often the strategy re-decides. Cadence is in
the spec rather than in the harness because it moves results more than any
signal parameter, so two runs of one fingerprint that differed in it could
disagree in sign. You can see all five in the JSON below.

In [2]:
spec = tf.StrategySpec.momentum(lookback_days=1.0, top_k=5)

print(spec.to_json())
print("fingerprint:", spec.fingerprint)

{
  "spec_version": 1,
  "signal": {
    "kind": "momentum",
    "lookback_days": 1.0
  },
  "portfolio": {
    "gross": 1.0,
    "top_k": 5
  },
  "execution": {
    "cadence": "step",
    "max_participation": 0.02
  },
  "seed": null
}
fingerprint: e6bbc35c6f0968b1f178e1f7ee926d449a8d3fa72440e476dd8b25c7a6a50895


It cannot express path dependence (stop losses, drawdown limits, anything
reading its own P&L history), conditional logic, or custom signals. Those
need a Python agent, which works everywhere a spec does. The cost is that
the result has to cite code at a commit instead of a fingerprint.

## Running an evaluation

`evaluate` runs every entrant against an identical market, which makes the
comparison exact. It is still only one market, which is the limitation the
rest of this notebook deals with.

In [3]:
entrants = {"mine": spec}
entrants.update(tf.baselines.reference_agents(seed=7))

scores = tf.evaluate(entrants, seed=7, universe=universe, days=10)

print(f"{'agent':16s} {'return':>9s} {'trades':>7s} {'impact bps':>11s}")
for name, s in sorted(scores.items(), key=lambda kv: -kv[1].return_pct):
    print(f"{name:16s} {s.return_pct:8.3f}% {s.trades:7d} {s.impact_bps:11.2f}")

agent               return  trades  impact bps
buy_and_hold        1.048%      40        0.06
oracle              0.324%     346        0.05
random             -2.453%    2388        0.03
mine               -2.593%     676        0.09
momentum           -2.593%     676        0.09
mean_reversion     -3.452%     684        0.08


The baselines are included deliberately: a return means nothing until you
know what buy-and-hold did on the same market. Here `buy_and_hold` made
1.048%, and every other entrant finished below it, `oracle` included.
`mine` lost 2.593% trading the same market.

## The market as benchmark

`tf.versus_buy_and_hold` gives each entrant's P&L less buy-and-hold's on
the same market. Every entrant starts with the same cash, so the difference
is in dollars.

In [4]:
versus = tf.versus_buy_and_hold(scores)
print("P&L over buy-and-hold:")
for name, value in sorted(versus.items(), key=lambda kv: -kv[1]):
    print(f"  {name:16s} {value:+12,.0f}")

print()
print("capture ratio:", tf.capture_ratio(scores) or "none reported")
print(tf.capture_withheld(scores))

P&L over buy-and-hold:
  oracle                 -7,243
  random                -35,004
  mine                  -36,408
  momentum              -36,408
  mean_reversion        -44,999

capture ratio: none reported
No capture ratio on pt-v20. Market moves there mostly stick: each shock moves fair value for good, so even perfect knowledge of the model's fair value leaves little edge. The Oracle made money in 10 of 14 test markets and its P&L follows the market's month, so a fraction of it would measure the month, not the agent. Compare against buy-and-hold instead.


`oracle` reads the simulator's own fair value, so it sees what no real
trader could. On pt-v19 and earlier the library also reports a capture
ratio, each P&L as a fraction of what `oracle` earned. On pt-v20, the
default, it reports none. Market moves there mostly stick: each shock moves
fair value for good, so even perfect knowledge of fair value leaves little
edge, and `oracle` finished $7,243 behind buy-and-hold here. A fraction of
its P&L would measure the market's month, and `tf.capture_withheld` returns
that reason as text.

On pt-v19 the same market draw has a capture ratio:

In [5]:
entrants = {"mine": spec}
entrants.update(tf.baselines.reference_agents(seed=7))
older = tf.evaluate(entrants, seed=7, universe=universe, days=10,
                    model="pt-v19")

print("capture ratio on pt-v19:")
for name, value in sorted(tf.capture_ratio(older).items(), key=lambda kv: -kv[1]):
    print(f"  {name:16s} {value:7.3f}")

capture ratio on pt-v19:
  buy_and_hold       0.054
  mean_reversion     0.050
  random            -0.516
  mine              -1.394
  momentum          -1.394


There `oracle` is a reference point rather than a ceiling. It gets the
same gross exposure and participation cap as every other entrant and spends
them on a naive equal-weight rule, so a strategy with a better portfolio
under the same constraint can score above 1.0. 1.0 is the reference's own
result, a scale to read against rather than a target.

## Why one seed isn't enough

The same comparison on three different market draws:

In [6]:
for seed in (7, 8, 9):
    e = {"mine": spec}
    e.update(tf.baselines.reference_agents(seed=seed))
    s = tf.evaluate(e, seed=seed, universe=universe, days=10)
    ranked = sorted(s.items(), key=lambda kv: -kv[1].return_pct)
    print(f"seed {seed}: " + "  ".join(f"{n}({v.return_pct:+.1f}%)"
                                        for n, v in ranked[:4]))

seed 7: buy_and_hold(+1.0%)  oracle(+0.3%)  random(-2.5%)  mine(-2.6%)


seed 8: buy_and_hold(+3.5%)  oracle(+1.8%)  mean_reversion(-2.3%)  random(-2.5%)


seed 9: oracle(-1.6%)  buy_and_hold(-2.1%)  mean_reversion(-2.7%)  random(-2.9%)


`buy_and_hold` holds the top place on seeds 7 and 8 and is second on seed
9, where every entrant lost money and `oracle` lost least. `mine` is fourth
on seed 7 and misses the top four on the other two draws. Where the
ordering changes between seeds, a single-seed leaderboard is telling you
about the seed rather than the strategies.

## Ranking across seeds

`rank` runs many seeds and compares entrants pairwise on the same market
draw. Pairing removes the market from the comparison, so a modest number of
seeds is still informative.

It takes a factory rather than built agents. Agents are stateful, and a
reused instance carries one market's history into the next with no visible
symptom. A spec avoids this because it is rebuilt for each seed.

In [7]:
def make_agents():
    e = {"mine": tf.StrategySpec.momentum(lookback_days=1.0, top_k=5)}
    e.update(tf.baselines.reference_agents(seed=0))
    return e

ranking = tf.rank(make_agents, seeds=[1, 2, 3, 4, 5, 6],
                  universe=universe, days=5)

print(f"{'agent':16s} {'vs buy-and-hold':>16s} {'ahead':>6s} "
      f"{'median P&L':>11s} {'first on':>9s}")
for r in ranking.table():
    print(f"{r.name:16s} {r.mean_excess_pnl:+16,.0f} "
          f"{r.seeds_ahead:4d}/{len(r.pnls)} {r.median_pnl:11,.0f} "
          f"{r.wins:5d}/{len(r.pnls)}")

agent             vs buy-and-hold  ahead  median P&L  first on
mean_reversion             +8,722    4/6      -5,454     4/6
buy_and_hold                   +0    0/6     -15,694     2/6
random                     -1,042    3/6     -14,016     0/6
mine                      -12,125    1/6     -26,546     0/6
momentum                  -12,125    1/6     -26,546     0/6


On pt-v20 the table sorts on `mean_excess_pnl`, each entrant's P&L less
buy-and-hold's averaged over the seeds, and `seeds_ahead` counts the seeds
where it finished ahead of buy-and-hold. On pt-v19 and earlier it sorts on
`pooled_capture`, total P&L over the reference's total. The `first on`
column counts seeds where an entrant finished top of the table, with
`oracle` held out of the running; it is a league position rather than a
head-to-head record. Here `mean_reversion` finished top on four of the six,
$8,722 a seed ahead of buy-and-hold. `mine` and `momentum` are the same
strategy and match in every column.

## The paired sign test

`decisive` is true only when one entrant won on every paired seed. That is
the strongest claim a sign test can make, and it needs no distributional
assumption.

In [8]:
for other in ("buy_and_hold", "random", "mean_reversion"):
    t = ranking.separation("mine", other)
    p = "n/a" if t["p_value"] is None else f"{t['p_value']:.3f}"
    print(f"  mine vs {other:16s} {t['wins_a']}-{t['wins_b']}"
          f"  ties {t['ties']}  decisive={t['decisive']}  p={p}")

  mine vs buy_and_hold     1-5  ties 0  decisive=False  p=0.219
  mine vs random           0-6  ties 0  decisive=True  p=0.031
  mine vs mean_reversion   0-6  ties 0  decisive=True  p=0.031


On pt-v20 `capture_withheld` is set and no capture is computed.
`unmeasurable` lists the seeds where capture could not be measured on a
preset that reports one, because the reference did not make money there
and a ratio against a negative denominator would flip the sign of the whole
table. They are reported rather than dropped: a result averaged over the
seeds that happened to work, while presenting itself as covering all of
them, is the quiet omission this library exists to avoid. Here the list is
empty, because nothing divides by the reference.

In [9]:
print("capture withheld:", ranking.capture_withheld is not None)
print("unmeasurable:", list(ranking.unmeasurable) or "none")
print()
print(ranking.report())

capture withheld: True
unmeasurable: none

6 seeds on universe 9be68b9bc37e... under model pt-v20
  mean_reversion    vs buy-and-hold       +8,722 a seed  ahead 4/6  wins 4/6
  buy_and_hold      the benchmark  median pnl      -15,694  wins 2/6
  random            vs buy-and-hold       -1,042 a seed  ahead 3/6  wins 0/6
  mine              vs buy-and-hold      -12,125 a seed  ahead 1/6  wins 0/6
  momentum          vs buy-and-hold      -12,125 a seed  ahead 1/6  wins 0/6
  No capture ratio on pt-v20. Market moves there mostly stick: each shock moves fair value for good, so even perfect knowledge of the model's fair value leaves little edge. The Oracle made money in 10 of 14 test markets and its P&L follows the market's month, so a fraction of it would measure the month, not the agent. Compare against buy-and-hold instead.


## A caveat

Good results here do not predict real returns. The price process comes from
a known model, so a strategy that fits its structure will look excellent
without telling you anything transferable. A strategy that fails here is
more informative: it broke against a live order book under honest impact
costs.

Next: **[3 · Why did the price move](03-why-did-the-price-move.ipynb)**.